In [1]:
import pandas as pd

### Importation des 3 ressources

In [2]:
joconde = pd.read_excel("../data/raw/joconde.xlsx")
museofile = pd.read_csv("../data/raw/museofile.csv", sep="|")
photo = pd.read_json("../data/raw/photographies_guerre.json")

### Exploration de la forme des données

In [6]:
print("Joconde :", joconde.shape)
print("   ", joconde.info())
print("museofile :", museofile.shape)
print("   ", museofile.info())
print("photo :", photo.shape)
print("   ", photo.info())

#display(joconde.head())
#display(museofile.head())
#display(photo.head())

Joconde : (4922, 64)
<class 'pandas.DataFrame'>
RangeIndex: 4922 entries, 0 to 4921
Data columns (total 64 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   reference      4922 non-null   str    
 1   ancien_depo    77 non-null     str    
 2   appellation    5 non-null      str    
 3   ancienne_ap    3269 non-null   str    
 4   ancienne_at    237 non-null    str    
 5   auteur         4798 non-null   str    
 6   bibliograph    2372 non-null   str    
 7   commentaire    1959 non-null   str    
 8   presence_im    4922 non-null   str    
 9   date_d_acqu    3734 non-null   str    
 10  date_de_dep    1144 non-null   str    
 11  decouverte_    300 non-null    str    
 12  denominatio    2904 non-null   str    
 13  lieu_de_dep    1145 non-null   str    
 14  description    266 non-null    str    
 15  mesures        4884 non-null   str    
 16  date_de_mis    4816 non-null   str    
 17  date_creati    940 non-null    str    
 18

Joconde contient 64 colonnes, certaines sont très peu renseignées voire inutilisées.  
Museofile contient 24 colonnes  
Photo contient 22 colonnes  

### Vérifications : 
1. Valeurs manquantes (colonnes vides ou très peu renseignées)
2. Doublons et id (référence unique / dupplication)
3. Valeurs et formats
4. Analyse des champs (significations des colonnes et repérer celles qui pourraient être communes aux 2 sources)


In [8]:
def colonnes_pauvres(df, nom, seuil=95):
    vides = (df.isna().mean() * 100).round(1)
    print(f"--- {nom} ---")
    print("Entièrement vides :", list(vides[vides == 100].index))
    print(f"Vides à plus de {seuil}% :", list(vides[(vides > seuil) & (vides < 100)].index))

colonnes_pauvres(joconde, "Joconde")
colonnes_pauvres(museofile, "Muséofile")
colonnes_pauvres(photo, "Photos")

--- Joconde ---
Entièrement vides : ['epoque', 'geographie_', 'lien_video', 'manquant_co', 'millesime_d_1', 'onomastique', 'periode_d_u', 'precisions__3', 'precisions__5', 'references_', 'references__1', 'reference_m', 'references__2', 'lien_site_a']
Vides à plus de 95% : ['ancien_depo', 'appellation', 'ancienne_at', 'manquant', 'precisions__1', 'periode_de_', 'lien_inha', 'source_de_l', 'utilisation']
--- Muséofile ---
Entièrement vides : []
Vides à plus de 95% : []
--- Photos ---
Entièrement vides : ['obj']
Vides à plus de 95% : ['nump', 'lieucor', 'autoeu', 'scle']


In [10]:
print("Joconde reference unique :", joconde["reference"].is_unique)
print("Museofile Identifiant unique :", museofile["Identifiant"].is_unique)
print("Photos ref unique :", photo["ref"].is_unique)

Joconde reference unique : True
Museofile Identifiant unique : True
Photos ref unique : True


In [13]:
print("=== JOCONDE ===")
print(joconde["domaine"].value_counts().head(10))
print(joconde["region"].value_counts().head(10))
print(joconde["auteur"].value_counts().head(10))

print("=== MUSEOFILE ===")
print(museofile["Categorie"].value_counts().head(10))
print(museofile["Region"].value_counts().head(10))

print("=== PHOTOS ===")
print(photo["pays"].value_counts().head(10))
print(photo["typdoc"].value_counts().head(10))
print(photo["datpv"].value_counts().head(10))

=== JOCONDE ===
domaine
['peinture']                             1336
['dessin']                               1196
['sculpture']                            1090
['estampe']                               642
['photographie']                          194
['sculpture', 'croyances - coutumes']      65
['dessin', 'cartes - plans']               51
['miniature']                              46
['photographie', 'architecture']           35
['estampe', 'cartes - plans']              35
Name: count, dtype: int64
region
Grand Est    4922
Name: count, dtype: int64
auteur
anonyme                        891
Dubois Paul (1829-1905)        302
DERAIN André                   156
SIMART Pierre Charles          101
LANCELOT Gustave                99
KOCH Paul                       96
GAUSSEN Alfred                  92
FICHOT Michel Charles           76
FINEZ Grégoire Nicolas          67
Le Gray Gustave (1820-1884)     55
Name: count, dtype: int64
=== MUSEOFILE ===
Categorie
musée en territoire rural   

## Conclusion de l'exploration des données

L'exploration a porté sur trois jeux de données issus de sources patrimoniales différentes :

| Source | Nombre d'enregistrements | Nombre de colonnes |
|---|---:|---:|
| Joconde | 4 922 | 64 |
| Muséofile | 1 216 | 24 |
| Photographies de guerre | 18 982 | 22 |

Les trois sources présentent des structures et des niveaux de détail différents.

### Qualité des données

L'analyse des valeurs manquantes a mis en évidence plusieurs particularités :

- **Joconde :** 14 colonnes sont entièrement vides et plusieurs autres sont très peu renseignées
- **Muséofile :** aucune colonne n'est entièrement vide. Certaines informations restent néanmoins incomplètes
- **Photographies de guerre :** une colonne est entièrement vide (`obj`) et plusieurs autres sont très peu renseignées

Les identifiants principaux des trois sources (`reference`, `Identifiant` et `ref`) sont uniques dans leurs jeux de données respectifs.

### Différences entre les sources

Les trois jeux de données ne décrivent pas le même type d'entité :

- Joconde décrit des œuvres et objets de collections de musées
- Muséofile décrit des musées et leurs caractéristiques.
- Photographies de guerre décrit des documents photographiques avec notamment des informations historiques et géographiques.

Certaines informations semblent communes, notamment la localisation géographique. Toutefois, les champs ne possèdent pas nécessairement la même signification ni le même format.